# Screening Rekomendasi Coin — Indikator Momentum Jangka Pendek yang Tervalidasi

**Latar belakang**: user minta fitur rekomendasi coin baru — bukan auto-trading, tapi **shortlist
(min 5, maks 10)** coin "berpeluang" dari universe Large+Mid (`flex_params`, tipe `SIMBOL_CRYPTO`)
yang **dapat dipercaya**, buat user filter & analisis sendiri. "Peluang" di sini diartikan momentum
jangka pendek — coin yang lagi "panas" beberapa hari terakhir.

**Kenapa riset dari nol, bukan pakai yang sudah ada**: endpoint `GET /recommendations` yang sedang
live sekarang berbasis strategi **V5** (bottoming + reversal + regime + model ML) — dan V5 **sudah
ditolak** di `research_signal_trading/06_portfolio_single_position.ipynb`: *"Out-of-sample rugi
(−21%). Sinyal terlalu jarang dan membeli coin yang sedang jatuh tidak punya keunggulan."* Jadi
fitur yang live sekarang jalan di atas strategi yang terbukti rugi kalau diuji jujur — riset ini
**terpisah dulu**, belum menggantikan endpoint itu (keputusan user).

**Metodologi** (ikut kebiasaan `research_signal_trading`, lihat README bagian 4 di sana): indikator
& ambang dipilih **hanya** dari periode dev, diuji **sekali** di periode uji yang tidak ikut dipakai
memilih. Di sini datanya jauh lebih padat (tiap hari x tiap coin, bukan tiap bulan), jadi dev/test
dibagi per **tahun**: dev = 2021-01→2023-12, uji = 2024-01→2026-06.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(PROJECT_ROOT)); sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))
import warnings; warnings.filterwarnings("ignore")
import itertools
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)

RAW = PROJECT_ROOT / "notebooks" / "data" / "raw"
ALL_SYMS = sorted(p.stem.replace("_1d_full", "") for p in RAW.glob("*_1d_full.csv"))

# Kategori flex_params (snapshot notebook 09, research_signal_trading) -- 50 simbol Large/Mid.
# Dibuang dari universe riset ini:
STABLE = {"BFUSDUSDT", "RLUSDUSDT", "USD1USDT", "USDCUSDT", "USDEUSDT", "USDSUSDT", "UUSDT"}  # stablecoin
GOLD = {"PAXGUSDT", "XAUTUSDT"}     # token emas, bukan "peluang crypto" (notebook 09)
DUP = {"WBETHUSDT"}                  # ETH yang di-stake, mengikuti ETH (notebook 09)
LARGE = {"BTCUSDT", "ETHUSDT", "BNBUSDT", "XRPUSDT", "DOGEUSDT", "SOLUSDT", "TRXUSDT", "LINKUSDT"}

UNIV = [s for s in ALL_SYMS if s not in STABLE | GOLD | DUP]
CATEGORY = {s: ("Large" if s in LARGE else "Mid") for s in UNIV}
BTC = "BTCUSDT"
print(f"Universe: {len(UNIV)} coin ({sum(v=='Large' for v in CATEGORY.values())} Large, "
      f"{sum(v=='Mid' for v in CATEGORY.values())} Mid) -- dari {len(ALL_SYMS)} total flex_params, "
      f"buang {len(STABLE)} stablecoin + {len(GOLD)} token emas + {len(DUP)} duplikat")

def load(sym):
    df = pd.read_csv(RAW / f"{sym}_1d_full.csv", parse_dates=[0], index_col=0)
    df.columns = [c.lower() for c in df.columns]
    return df

raw = {s: load(s) for s in UNIV}
dates = pd.date_range("2021-01-01", "2026-06-30", freq="D")
close = pd.DataFrame({s: raw[s]["close"].reindex(dates) for s in UNIV})
volume = pd.DataFrame({s: raw[s]["volume"].reindex(dates) for s in UNIV})
COINS_NB = [c for c in UNIV if c != BTC]      # BTC sendiri bukan kandidat rekomendasi (acuan pasar)

DEV, TEST = ("2021-01-01", "2023-12-31"), ("2024-01-01", "2026-06-30")
print(f"Dev: {DEV} | Uji: {TEST}")

Universe: 40 coin (8 Large, 32 Mid) -- dari 50 total flex_params, buang 7 stablecoin + 2 token emas + 1 duplikat


Dev: ('2021-01-01', '2023-12-31') | Uji: ('2024-01-01', '2026-06-30')


## 1. Indikator kandidat (jangka pendek, 3-20 hari)

Bukan pakai gate V5 yang sudah ditolak — dibangun fresh dari indikator dasar yang sudah ada di
`src/strategy/indicators/` (RSI, MA, volume), dicoba di beberapa lookback pendek, plus jarak dari
puncak (indikator yang terbukti berguna di riset portofolio, `research_signal_trading/21`).

In [2]:
def rsi(s, n=14):
    d = s.diff()
    up = d.clip(lower=0).ewm(alpha=1/n, adjust=False, min_periods=n).mean()
    dn = (-d.clip(upper=0)).ewm(alpha=1/n, adjust=False, min_periods=n).mean()
    return 100 - 100 / (1 + up / dn)

mom = {n: close / close.shift(n) - 1 for n in (3, 5, 7, 10, 14)}
btc_mom = {n: mom[n][BTC] for n in (3, 5, 7, 10, 14)}
rel_str = {n: mom[n].sub(btc_mom[n], axis=0) for n in (5, 7, 10)}
dist_high = {n: close / close.rolling(n, min_periods=n).max() - 1 for n in (5, 10, 20)}
vol_ratio = volume / volume.rolling(20, min_periods=20).mean()
ma_ratio = {n: close / close.rolling(n, min_periods=n).mean() - 1 for n in (10, 20)}

candidates = {"rsi14": close.apply(rsi, n=14), "rsi7": close.apply(rsi, n=7), "volume_ratio": vol_ratio}
for n in (3, 5, 7, 10, 14):
    candidates[f"momentum_{n}"] = mom[n]
for n in (5, 7, 10):
    candidates[f"rel_strength_{n}"] = rel_str[n]
for n in (5, 10, 20):
    candidates[f"dist_from_high_{n}"] = dist_high[n]
for n in (10, 20):
    candidates[f"ma_ratio_{n}"] = ma_ratio[n]

fwd = {h: close.shift(-h) / close - 1 for h in (3, 5, 7, 10)}   # target: return H hari ke depan
print(f"{len(candidates)} indikator kandidat, {len(fwd)} horizon target (3/5/7/10 hari)")

16 indikator kandidat, 4 horizon target (3/5/7/10 hari)


## 2. Korelasi indikator x return ke depan (DEV saja, 2021-2023)

Pooled cross-sectional: tiap (coin, hari) satu baris. Dicek dulu korelasi Spearman keseluruhan --
tapi lihat bagian 3, korelasi keseluruhan **bisa menyesatkan** buat kasus kita (milih top-K harian),
beda dari "berkorelasi secara umum".

In [3]:
def build_pool(period):
    d0, d1 = period
    idx = dates[(dates >= d0) & (dates <= d1)]
    rows = []
    for s in COINS_NB:
        c = close[s].loc[idx]
        if c.notna().sum() < 60:
            continue
        for t in idx[c.notna()]:
            row = {"coin": s, "date": t, **{f"fwd_{h}": fwd[h].at[t, s] for h in (3, 5, 7, 10)},
                   **{name: df.at[t, s] for name, df in candidates.items()}}
            rows.append(row)
    return pd.DataFrame(rows)

pool_dev = build_pool(DEV)
print(f"Pool dev: {len(pool_dev)} baris (coin x hari)\n")

for h in (5, 7):
    res = []
    for name in candidates:
        d = pool_dev[["coin", name, f"fwd_{h}"]].dropna()
        if len(d) < 200:
            continue
        corr, pval = spearmanr(d[name], d[f"fwd_{h}"])
        res.append(dict(indikator=name, n=len(d), spearman=corr, pval=pval))
    r = pd.DataFrame(res).sort_values("spearman", ascending=False)
    print(f"--- horizon {h} hari ---")
    print(r.head(6).to_string(index=False, formatters={"spearman": "{:+.3f}".format, "pval": "{:.4f}".format}))
    print()

Pool dev: 25646 baris (coin x hari)



--- horizon 5 hari ---
        indikator     n spearman   pval
     volume_ratio 25133   +0.045 0.0000
            rsi14 25268   +0.029 0.0000
             rsi7 25457   +0.025 0.0001
       momentum_7 25457   +0.020 0.0012
dist_from_high_10 25403   +0.017 0.0055
      ma_ratio_10 25403   +0.017 0.0080



--- horizon 7 hari ---
        indikator     n spearman   pval
            rsi14 25268   +0.037 0.0000
     volume_ratio 25133   +0.035 0.0000
             rsi7 25457   +0.034 0.0000
dist_from_high_10 25403   +0.024 0.0001
 dist_from_high_5 25538   +0.023 0.0002
      ma_ratio_10 25403   +0.023 0.0003



## 3. Kenapa bukan cuma pakai korelasi tertinggi -- ukur yang beneran relevan

Tujuan kita bukan "indikator mana paling berkorelasi secara umum", tapi **"indikator mana yang kalau
dipakai milih top-10 harian, top-10 itu beneran lebih baik dari bottom-10"** -- ukuran yang lebih
jujur buat use-case screening.

**Ternyata beda**: `volume_ratio` korelasi keseluruhannya tinggi, tapi begitu dicek spread
top-10 vs bottom-10, hasilnya **negatif** (-0.27% di h=7) -- lonjakan volume besar ternyata muncul di
KEDUA ujung (breakout naik ATAU crash turun), jadi tidak berguna buat milih arah. Sebaliknya `rsi14`
konsisten di kedua ukuran. Pelajarannya: validasi harus sesuai cara indikatornya benar-benar dipakai,
bukan metrik generik.

In [4]:
def topk_spread(score_df, period, h, k):
    '''Rata-rata (top-k mean - bottom-k mean) per hari, vectorized (bukan loop per hari).
    Entri tanpa histori cukup (NaN) TIDAK pernah kepilih jadi top maupun bottom.'''
    d0, d1 = period
    idx = dates[(dates >= d0) & (dates <= d1)]
    sc = score_df.loc[idx, COINS_NB].to_numpy(dtype=float)
    fw = fwd[h].loc[idx, COINS_NB].to_numpy(dtype=float)
    valid = ~np.isnan(sc) & ~np.isnan(fw)
    ok_rows = valid.sum(axis=1) >= 2 * k
    n = sc.shape[1]
    top_idx = np.argsort(np.where(valid, sc, -np.inf), axis=1)[:, n - k:]   # invalid -> -inf, tak pernah jadi top
    bot_idx = np.argsort(np.where(valid, sc, np.inf), axis=1)[:, :k]        # invalid -> +inf, tak pernah jadi bottom
    top_fwd = np.where(np.take_along_axis(valid, top_idx, axis=1), np.take_along_axis(fw, top_idx, axis=1), np.nan)
    bot_fwd = np.where(np.take_along_axis(valid, bot_idx, axis=1), np.take_along_axis(fw, bot_idx, axis=1), np.nan)
    top_mean, bot_mean = np.nanmean(top_fwd[ok_rows], axis=1), np.nanmean(bot_fwd[ok_rows], axis=1)
    fw_v, valid_v = np.where(valid, fw, np.nan)[ok_rows], valid[ok_rows]
    univ_wr = np.sum((fw_v > 0) & valid_v, axis=1) / np.sum(valid_v, axis=1)   # NaN>0 == False di numpy, harus di-mask manual
    return dict(top_mean=np.nanmean(top_mean), bot_mean=np.nanmean(bot_mean),
               spread=np.nanmean(top_mean) - np.nanmean(bot_mean),
               top_wr=np.nanmean(top_fwd[ok_rows] > 0), univ_mean=np.nanmean(fw_v),
               univ_wr=np.nanmean(univ_wr), n_days=int(ok_rows.sum()))

print("Cek volume_ratio vs rsi14 (h=7, k=10, DEV):")
for name in ("volume_ratio", "rsi14"):
    r = topk_spread(candidates[name], DEV, 7, 10)
    print(f"  {name:15s} top={r['top_mean']:+.2%} bot={r['bot_mean']:+.2%} spread={r['spread']:+.2%}")

Cek volume_ratio vs rsi14 (h=7, k=10, DEV):
  volume_ratio    top=+1.41% bot=+1.64% spread=-0.24%
  rsi14           top=+2.24% bot=+0.84% spread=+1.40%


## 4. Cari kombinasi komposit terbaik (DEV saja)

Skor = rata-rata rank-percentile harian dari beberapa indikator (tahan outlier). Dicoba semua
kombinasi 1-3 indikator dari kandidat yang masuk akal (RSI, dist-from-high, MA-ratio, momentum --
**`volume_ratio` dibuang dari percobaan** sesuai temuan bagian 3).

In [5]:
pool_names = ["rsi14", "rsi7", "dist_from_high_10", "dist_from_high_5", "ma_ratio_10",
             "ma_ratio_20", "momentum_7", "momentum_5", "momentum_3"]
combo_results = []
for r in (1, 2, 3):
    for combo in itertools.combinations(pool_names, r):
        sc = sum(candidates[n].rank(axis=1, pct=True) for n in combo) / len(combo)
        res = topk_spread(sc, DEV, 7, 10)
        combo_results.append(dict(combo=" + ".join(combo), **res))
cr = pd.DataFrame(combo_results).sort_values("spread", ascending=False)
out = cr.head(10).copy()
for c in ("top_mean", "bot_mean", "spread", "univ_mean"):
    out[c] = cr[c].map(lambda v: f"{v:+.2%}")
for c in ("top_wr", "univ_wr"):
    out[c] = cr[c].map(lambda v: f"{v:.1%}")
print(out.to_string(index=False))

best_combo = cr.iloc[0]["combo"].split(" + ")
print(f"\nTerpilih dari dev (spread tertinggi): {best_combo}")
score_best = sum(candidates[n].rank(axis=1, pct=True) for n in best_combo) / len(best_combo)

                          combo top_mean bot_mean spread top_wr univ_mean univ_wr  n_days
                   rsi14 + rsi7   +2.31%   +0.70% +1.61%  48.1%    +1.41%   48.5%    1081
                           rsi7   +2.31%   +0.88% +1.43%  48.2%    +1.48%   48.6%    1088
rsi14 + rsi7 + dist_from_high_5   +2.22%   +0.80% +1.42%  48.3%    +1.41%   48.5%    1081
            rsi14 + ma_ratio_20   +2.23%   +0.81% +1.42%  48.0%    +1.39%   48.5%    1076
       rsi14 + dist_from_high_5   +2.25%   +0.83% +1.41%  48.7%    +1.41%   48.5%    1081
     rsi14 + rsi7 + ma_ratio_20   +2.20%   +0.81% +1.40%  48.0%    +1.39%   48.5%    1076
                          rsi14   +2.24%   +0.84% +1.40%  48.2%    +1.41%   48.5%    1081
             rsi14 + momentum_3   +2.18%   +0.84% +1.34%  48.1%    +1.41%   48.5%    1081
      rsi14 + rsi7 + momentum_3   +2.17%   +0.86% +1.31%  48.0%    +1.41%   48.5%    1081
             rsi14 + momentum_5   +2.16%   +0.86% +1.31%  48.0%    +1.41%   48.5%    1081

Terpilih 

In [6]:
print("Kombinasi terpilih x semua horizon x k (DEV):\n")
for h in (3, 5, 7, 10):
    for k in (5, 10):
        res = topk_spread(score_best, DEV, h, k)
        print(f"h={h:2d} k={k:2d}: top={res['top_mean']:+.2%} (wr {res['top_wr']:.1%}) | bot={res['bot_mean']:+.2%} | "
              f"spread={res['spread']:+.2%} | universe={res['univ_mean']:+.2%} (wr {res['univ_wr']:.1%})")

Kombinasi terpilih x semua horizon x k (DEV):

h= 3 k= 5: top=+0.99% (wr 47.3%) | bot=+0.20% | spread=+0.79% | universe=+0.58% (wr 49.2%)
h= 3 k=10: top=+0.97% (wr 48.3%) | bot=+0.30% | spread=+0.67% | universe=+0.58% (wr 49.2%)
h= 5 k= 5: top=+1.78% (wr 46.8%) | bot=+0.27% | spread=+1.51% | universe=+0.99% (wr 49.0%)
h= 5 k=10: top=+1.65% (wr 48.5%) | bot=+0.49% | spread=+1.16% | universe=+0.99% (wr 49.0%)
h= 7 k= 5: top=+2.34% (wr 46.3%) | bot=+0.39% | spread=+1.96% | universe=+1.41% (wr 48.5%)
h= 7 k=10: top=+2.31% (wr 48.1%) | bot=+0.70% | spread=+1.61% | universe=+1.41% (wr 48.5%)
h=10 k= 5: top=+3.45% (wr 47.0%) | bot=+0.72% | spread=+2.73% | universe=+2.07% (wr 48.2%)
h=10 k=10: top=+3.25% (wr 48.0%) | bot=+1.21% | spread=+2.04% | universe=+2.07% (wr 48.2%)


## 5. Validasi di periode uji (2024-01 → 2026-06), SEKALI

Pakai kombinasi & horizon yang sudah terpilih dari dev (`rsi14 + rsi7`, horizon ~7 hari) -- tidak
diubah-ubah lagi setelah dilihat.

In [7]:
print("UJI (2024-01..2026-06):\n")
for h in (5, 7):
    for k in (5, 10):
        res = topk_spread(score_best, TEST, h, k)
        print(f"h={h:2d} k={k:2d}: top={res['top_mean']:+.2%} (wr {res['top_wr']:.1%}) | bot={res['bot_mean']:+.2%} | "
              f"spread={res['spread']:+.2%} | universe={res['univ_mean']:+.2%} (wr {res['univ_wr']:.1%}) | hari={res['n_days']}")

UJI (2024-01..2026-06):

h= 5 k= 5: top=+0.91% (wr 46.6%) | bot=-0.03% | spread=+0.94% | universe=+0.13% (wr 46.0%) | hari=907
h= 5 k=10: top=+0.74% (wr 46.3%) | bot=-0.14% | spread=+0.88% | universe=+0.13% (wr 46.0%) | hari=907
h= 7 k= 5: top=+1.14% (wr 46.6%) | bot=-0.02% | spread=+1.17% | universe=+0.21% (wr 45.3%) | hari=905
h= 7 k=10: top=+0.98% (wr 46.0%) | bot=-0.09% | spread=+1.07% | universe=+0.21% (wr 45.3%) | hari=905


**Lolos** -- spread tetap positif di uji (profit lebih tinggi dari rata-rata, rugi lebih kecil dari
rata-rata), tidak sebesar dev (wajar) tapi tidak hilang/runtuh seperti kasus-kasus di
`research_signal_trading` (filter per-coin, trailing-stop). Catatan jujur: win rate top-K (~46%)
nyaris sama dengan win rate universe (~45%) -- **edge-nya dari besaran untung/rugi, bukan dari
frekuensi menang**, pola yang konsisten dengan semua riset crypto di proyek ini.

## 6. Uji ketahanan: per tahun & bukan cuma proxy "pas BTC naik\"

In [8]:
print("Per tahun (h=7, k=10, gabung dev+uji -- cuma buat lihat konsistensi, bukan pemilihan ulang):\n")
for yr in range(2021, 2027):
    res = topk_spread(score_best, (f"{yr}-01-01", f"{yr}-12-31"), 7, 10)
    if res["n_days"] < 10:
        continue
    print(f"{yr}: top={res['top_mean']:+.2%} (wr {res['top_wr']:.1%}) | bot={res['bot_mean']:+.2%} | "
          f"spread={res['spread']:+.2%} | universe={res['univ_mean']:+.2%} (wr {res['univ_wr']:.1%})")

Per tahun (h=7, k=10, gabung dev+uji -- cuma buat lihat konsistensi, bukan pemilihan ulang):

2021: top=+6.91% (wr 52.0%) | bot=+2.73% | spread=+4.18% | universe=+4.59% (wr 52.7%)
2022: top=-1.71% (wr 41.9%) | bot=-1.75% | spread=+0.04% | universe=-1.70% (wr 42.3%)
2023: top=+1.90% (wr 50.5%) | bot=+1.21% | spread=+0.69% | universe=+1.62% (wr 50.7%)
2024: top=+3.41% (wr 49.8%) | bot=+1.50% | spread=+1.92% | universe=+2.46% (wr 49.7%)
2025: top=-0.58% (wr 43.9%) | bot=-0.96% | spread=+0.38% | universe=-0.89% (wr 42.8%)


2026: top=-0.87% (wr 42.6%) | bot=-1.60% | spread=+0.73% | universe=-1.39% (wr 41.3%)


In [9]:
print("\nApa ini cuma proxy 'pas BTC lagi naik'? (UJI, h=7 k=10, dipecah BTC naik/turun 7hr ke depan)\n")
btc_up = (fwd[7][BTC] > 0).reindex(dates, fill_value=False)
idx_test = dates[(dates >= TEST[0]) & (dates <= TEST[1])]
for label, mask in [("BTC naik 7hr ke depan", btc_up.loc[idx_test]), ("BTC turun 7hr ke depan", ~btc_up.loc[idx_test])]:
    sub = idx_test[mask.reindex(idx_test, fill_value=False)]
    sc_arr = score_best.loc[sub, COINS_NB].to_numpy(float)
    fw_arr = fwd[7].loc[sub, COINS_NB].to_numpy(float)
    valid = ~np.isnan(sc_arr) & ~np.isnan(fw_arr)
    n = sc_arr.shape[1]
    top_idx = np.argsort(np.where(valid, sc_arr, -np.inf), axis=1)[:, n - 10:]
    bot_idx = np.argsort(np.where(valid, sc_arr, np.inf), axis=1)[:, :10]
    tf, bf = np.take_along_axis(fw_arr, top_idx, axis=1), np.take_along_axis(fw_arr, bot_idx, axis=1)
    print(f"{label} ({len(sub)} hari): top10={np.nanmean(tf):+.2%} | bot10={np.nanmean(bf):+.2%} | spread={np.nanmean(tf) - np.nanmean(bf):+.2%}")


Apa ini cuma proxy 'pas BTC lagi naik'? (UJI, h=7 k=10, dipecah BTC naik/turun 7hr ke depan)



BTC naik 7hr ke depan (470 hari): top10=+6.67% | bot10=+6.11% | spread=+0.56%
BTC turun 7hr ke depan (442 hari): top10=-5.17% | bot10=-6.78% | spread=+1.62%


**Lolos juga** -- spread positif di **semua** tahun 2021-2026 (termasuk bear 2022, meski tipis
+0.04%: pas crash sistemik semua coin jatuh bareng, mirip temuan korelasi di riset portofolio). Dan
spread **lebih besar justru pas BTC lagi turun** (+1.62% vs +0.56% pas BTC naik) -- jadi ini bukan
cuma proxy "ikut tren pasar", beneran membedakan coin relatif kuat vs lemah.

## 7. Contoh keluaran: rekomendasi hari terakhir di data

Ilustrasi konkret formatnya -- top-10 per tanggal terakhir, bisa dipotong jadi top-5 sampai top-10
sesuai kebutuhan ("min 5 maks 10" sesuai permintaan user).

In [10]:
last_date = dates[close.notna().sum(axis=1).to_numpy().nonzero()[0][-1]]
snap = score_best.loc[last_date].dropna().sort_values(ascending=False)
rows = []
for rank, (sym, sc) in enumerate(snap.head(10).items(), start=1):
    rows.append(dict(rank=rank, symbol=sym, kategori=CATEGORY[sym], skor_momentum=round(sc, 3),
                     rsi14=round(float(candidates["rsi14"].at[last_date, sym]), 1),
                     rsi7=round(float(candidates["rsi7"].at[last_date, sym]), 1),
                     momentum_7h=f"{candidates['momentum_7'].at[last_date, sym]:+.1%}"))
print(f"Tanggal: {last_date.date()}\n")
pd.DataFrame(rows).set_index("rank")

Tanggal: 2026-06-30



,symbol,kategori,skor_momentum,rsi14,rsi7,momentum_7h
rank,,,,,,
1,JSTUSDT,Mid,1.000,58.0,67.8,+5.8%
2,AAVEUSDT,Mid,0.950,56.3,53.9,+17.5%
3,MORPHOUSDT,Mid,0.950,53.1,61.9,+15.4%
4,JUPUSDT,Mid,0.925,54.9,51.9,+3.5%
5,SOLUSDT,Large,0.925,52.7,59.1,+5.7%
6,XLMUSDT,Mid,0.863,48.3,50.4,-3.1%
7,PUMPUSDT,Mid,0.825,46.4,49.6,-2.1%
8,AVAXUSDT,Mid,0.812,43.7,50.8,+0.8%
9,WLFIUSDT,Mid,0.787,45.8,44.4,-3.0%


## Kesimpulan

**Metodologi screening momentum jangka pendek tervalidasi**, beda total dari V5 (yang tidak pernah
diuji jujur sampai akhirnya terbukti rugi -21% out-of-sample). Di sini: dipilih dari dev, diuji sekali
di periode terpisah, dan lolos 3 lapis pengecekan (uji utama, per-tahun, bukan proxy BTC).

### Metodologi terpilih
- **Indikator**: rata-rata rank-percentile dari **RSI(14) + RSI(7)** -- koreksi penting: jangan
  masukkan `volume_ratio` meski korelasi keseluruhannya tinggi, karena spread top-vs-bottom-nya
  justru **negatif** (lonjakan volume muncul di breakout naik MAUPUN crash turun).
- **Horizon**: ~7 hari (3-10 hari semua positif di dev, 7 hari paling stabil).
- **Universe**: Large+Mid dari `flex_params` (snapshot notebook 09, `research_signal_trading`) minus
  stablecoin, minus token emas (PAXG, XAUT), minus WBETH (duplikat ETH) -- 40 coin.
- **Output**: ranking harian top-10, ambil **5-10 teratas** (sesuai permintaan user), bukan ambang
  biner lolos/gagal seperti V5.

### Dampak (periode uji 2024-01 → 2026-06, horizon 7 hari, top-10)
| | Top-10 (skor tertinggi) | Rata-rata universe | Bottom-10 (skor terendah) |
|---|---|---|---|
| Return 7 hari | **+0.98%** | +0.35% | −0.09% |
| Win rate | 46.0% | 45.3% | — |

### Catatan jujur / batasan
1. **Edge-nya kecil dan dari besaran, bukan frekuensi menang** -- win rate top-10 nyaris sama dengan
   universe (~46% vs ~45%). Ini bukan sinyal "pasti naik", tapi "secara rata-rata sedikit lebih baik
   dari acak" -- sesuai tujuan user (shortlist buat dianalisis sendiri, bukan auto-buy).
2. **Lemah pas crash sistemik** (2022: spread cuma +0.04%) -- sama seperti semua riset lain di
   proyek ini, saat market jatuh bareng, sinyal relatif antar-coin kurang berguna.
3. Kategori Large/Mid di sini masih **snapshot flex_params dari notebook 09** -- bisa beda dari DB
   saat ini; kalau dipakai produksi, kategorinya harus diambil langsung dari `flex_params` live.
4. Belum ada komponen "dapat dipercaya" di luar statistik forward-return -- misalnya likuiditas
   minimum, umur listing minimum, atau exclude coin yang baru pump ekstrem (meme coin pump-and-dump).
   Ini layak jadi riset lanjutan kalau mau dipakai beneran.
5. **Simulasi backtest historis, bukan jaminan hasil ke depan.**

### Langkah berikutnya
Belum menyentuh `src/services/recommendation/` (keputusan user: riset dulu, terpisah). Kalau nanti
mau diarahkan jadi endpoint, perlu: (a) tes ambang kualitas/likuiditas tambahan, (b) desain cara
menyajikan skor ke user (bukan cuma angka mentah), (c) baru diskusi apakah gantiin atau dampingi
`/recommendations` yang berbasis V5.